# [LAB-04] 데이터 다루기

## 8. 데이터 재구조화

### #01 준비작업

1. 라이브러리 참조

In [3]:
from jussam import load_data
from pandas import DataFrame, pivot_table, melt

2. 샘플 데이터 가져오기

In [4]:
origin = load_data("city_people")
origin

📚 서울, 인천, 부산에서 2005년, 2010년 2015년에 조사한 가상의 인구수 데이터(인덱스와 메타데이터 없음)


,도시,연도,인구,지역
0,서울,2015,9904312,수도권
1,서울,2010,9631482,수도권
2,서울,2005,9762546,수도권
3,부산,2015,3448737,경상권
4,부산,2010,3393191,경상권
5,부산,2005,3512547,경상권
6,인천,2015,2890451,수도권
7,인천,2010,2632035,수도권


### #02 피벗 테이블

1. 기본 사용 방법 (도시와 연도에 따른 인구수 재배치)

In [5]:
pivot1 = pivot_table(origin,                # 피벗할 데이터프레임
                     index= '도시',         # 행 위치에 들어갈 열
                     columns= '연도',       # 열 위치에 들어갈 열
                     values= '인구')        # 데이터로 사용할 열
print(type(pivot1))
pivot1

<class 'pandas.DataFrame'>


연도,2005,2010,2015
도시,,,
부산,3512547.000,3393191.000,3448737.000
서울,9762546.000,9631482.000,9904312.000
인천,NaN,2632035.000,2890451.000


2. 중복 데이터의 집계 방법 지정하기

In [6]:
pivot_table(origin,                # 피벗할 데이터프레임
            index= '지역',         # 행 위치에 들어갈 열
            columns= '연도',       # 열 위치에 들어갈 열
            values= '인구',        # 데이터로 사용할 열
            aggfunc= 'sum')        # 데이터가 두 개 이상일 경우 집계함수 지정

연도,2005,2010,2015
지역,,,
경상권,3512547,3393191,3448737
수도권,9762546,12263517,12794763


3. 두 개 이상의 집계 함수 지정

In [7]:
pivot_table(origin,
            index= '지역',
            columns= '연도',
            values= '인구',
            aggfunc= ['sum', 'mean'])

sum                            mean                        
연도      2005      2010      2015        2005        2010        2015
지역                                                                  
경상권  3512547   3393191   3448737 3512547.000 3393191.000 3448737.000
수도권  9762546  12263517  12794763 9762546.000 6131758.500 6397381.500

4. 복수 인덱스 지정

In [8]:
pivot_table(origin,
            index= ['지역', '연도'],
            columns= '도시',
            values= '인구',
            aggfunc= ['sum', 'mean'])

sum                                mean              \
도시                부산          서울          인천          부산          서울   
지역  연도                                                                 
경상권 2005 3512547.000         NaN         NaN 3512547.000         NaN   
    2010 3393191.000         NaN         NaN 3393191.000         NaN   
    2015 3448737.000         NaN         NaN 3448737.000         NaN   
수도권 2005         NaN 9762546.000         NaN         NaN 9762546.000   
    2010         NaN 9631482.000 2632035.000         NaN 9631482.000   
    2015         NaN 9904312.000 2890451.000         NaN 9904312.000   

                      
도시                인천  
지역  연도                
경상권 2005         NaN  
    2010         NaN  
    2015         NaN  
수도권 2005         NaN  
    2010 2632035.000  
    2015 2890451.000

### #03 Melt

1. 샘플 피벗 테이블 생성

In [9]:
pivot_df = pivot_table(origin,
                       index= '연도',
                       columns= '지역',
                       values= '인구',
                       aggfunc= 'mean')
pivot_df

지역,경상권,수도권
연도,,
2005,3512547.000,9762546.000
2010,3393191.000,6131758.500
2015,3448737.000,6397381.500


2. 피벗 테이블 분리 1단계: 데이터 프레임의 인덱스를 일반 컬럼으로 설정

In [10]:
pivot_df2 = pivot_df.reset_index()
pivot_df2

지역,연도,경상권,수도권
0,2005,3512547.000,9762546.000
1,2010,3393191.000,6131758.500
2,2015,3448737.000,6397381.500


3. 피벗 테이블 분리 2단계: Melt 처리

In [11]:
mdf = melt(pivot_df2, id_vars=['연도'], value_vars=['경상권', '수도권'])
mdf

,연도,지역,value
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


4. 피벗 테이블 분리 시 필드 이름 지정하기

In [12]:
mdf = melt(pivot_df2, id_vars=['연도'], value_vars=['경상권', '수도권'],
           var_name='구분', value_name='인구수')
mdf

,연도,구분,인구수
0,2005,경상권,3512547.000
1,2010,경상권,3393191.000
2,2015,경상권,3448737.000
3,2005,수도권,9762546.000
4,2010,수도권,6131758.500
5,2015,수도권,6397381.500


### #04 Stack, Unstack 실습을 위한 준비작업

1. 샘플 데이터 가져오기

In [13]:
origin = load_data("body_size")
origin

📚 어느 학급 학생들의 이름, 성별, 키, 몸무게에 대한 가상의 데이터

    field    description
--  -------  -------------
 0  name     이름
 1  sex      성별
 2  height   키
 3  weight   몸무게



,sex,height,weight
name,,,
Lee,M,175,98.000
Park,F,167,48.000
Hong,M,180,NaN
Kim,F,162,55.000
Nam,M,172,85.000


### #05 Stack

1. 기본 사용 방법

In [14]:
st1 = origin.stack()
print(type(st1))
st1

<class 'pandas.Series'>


name        
Lee   sex           M
      height      175
      weight   98.000
Park  sex           F
      height      167
      weight   48.000
Hong  sex           M
      height      180
      weight      NaN
Kim   sex           F
      height      162
      weight   55.000
Nam   sex           M
      height      172
      weight   85.000
dtype: object

2. Stack 결과를 DataFrame으로 생성하기

In [15]:
df1 = DataFrame(origin.stack())
df1

0
name              
Lee  sex         M
     height    175
     weight 98.000
Park sex         F
     height    167
     weight 48.000
Hong sex         M
     height    180
     weight    NaN
Kim  sex         F
     height    162
     weight 55.000
Nam  sex         M
     height    172
     weight 85.000

### #06 Unstack

1. Stack 결과를 원래대로 되돌린다

In [16]:
ust = df1.unstack()
ust

0              
     sex height weight
name                  
Lee    M    175 98.000
Park   F    167 48.000
Hong   M    180    NaN
Kim    F    162 55.000
Nam    M    172 85.000

---

## 연습문제

### 1. 학생 흡연율 데이터 전처리
students_smoke 데이터 셋은 한 도시의 20개 중·고교를 대상으로 조사한 흡연율 자료이다.   
도시와 농촌별에 따른 남여 평균 흡연율을 확인할 수 있는 데이터 피벗테이블을 구성하시오.

In [17]:
# 1. 라이브러리 참조
from jussam import load_data
from pandas import DataFrame, pivot_table, melt

# 2. 데이터 불러오기
smoke = load_data('students_smoke')

# 3. 피벗
pivot1 = pivot_table(smoke,
                     index= '지역',
                     columns= '구성',
                     values= '흡연율',
                     aggfunc='mean')
pivot1

📚 한 도시의 20개 중,고교를 대상으로 조사한 흡연율 자료 (인덱스와 메타데이터 없음)


구성,공학,남,여
지역,,,
농촌,0.757,0.745,0.463
도시,0.680,0.792,0.507


### 2. 동네 아이스크림 가게 매출 분석 챌린지!

**당신은 최근 동네에 새로 생긴 '달콤 스쿱' 아이스크림 가게의 데이터 분석을 의뢰 받았습니다.**
- 사장님은 여름 시즌을 맞아 어떤 맛 아이스크림이 인기가 많은지, 날짜별 판매 차이는 어떠한지 궁금해하며, 효율적인 재고 관리를 위한 데이터 분석을 당신에게 의뢰했습니다.
- 당신에게 주어진 icecream_sales 데이터셋을 사용하여 다음의 의뢰를 완수하세요.   

**요구사항**
1. 데이터의 전체적인 구조와 정보를 확인하고, 결측치가 있는지 확인하세요.
   만약 결측치가 있다면, 해당 행을 제거하세요.
2. 판매 날짜를 인덱스로, 아이스크림 맛을 컬럼으로 하여 각 맛의 일별 판매량을 보여주는 피벗 테이블을 만드세요.
3. 사장님이 다른 분석 툴에서 사용하고 싶다며, 위에서 만든 피벗 테이블을 다시 긴 형식(Long format)의 데이터로 분리해달라고 요청했습니다. 컬럼 이름은 원본 이름을 그대로 유지해야 합니다.

In [18]:
# 1. 라이브러리 참조
from jussam import load_data
from pandas import DataFrame, pivot_table, melt

# 2. 데이터 불러오기
icecream = load_data('icecream_sales')

# 3. 결측치 확인
na = icecream.isna()
na.sum()

📚 '달콤 스쿱' 아이스크림 가게의 매출 데이터 (인덱스 없음)


Date        0
Flavor      0
Topping     1
Price       0
Quantity    1
dtype: int64

In [19]:
# 4. 데이터 정제
icecream1 = icecream.dropna()
icecream1

,Date,Flavor,Topping,Price,Quantity
0,2023-07-01,초콜릿,아몬드,3500,20.000
1,2023-07-01,바닐라,초코시럽,3000,25.000
2,2023-07-01,딸기,연유,3200,18.000
3,2023-07-02,민트초코,초코칩,3800,15.000
4,2023-07-02,초콜릿,아몬드,3500,22.000
6,2023-07-03,딸기,연유,3200,25.000
8,2023-07-03,바닐라,초코시럽,3000,28.000
9,2023-07-04,초콜릿,아몬드,3500,18.000
10,2023-07-04,딸기,연유,3200,22.000
11,2023-07-04,민트초코,초코칩,3800,17.000


In [20]:
# 5. 피벗 테이블
pivot1 = pivot_table(icecream1,
                     index= 'Date',
                     columns= 'Flavor',
                     values= 'Quantity')
pivot1

Flavor,딸기,민트초코,바닐라,초콜릿
Date,,,,
2023-07-01,18.000,NaN,25.000,20.000
2023-07-02,NaN,15.000,NaN,22.000
2023-07-03,25.000,NaN,28.000,NaN
2023-07-04,22.000,17.000,NaN,18.000


In [21]:
# 6. 피벗 테이블 분리
pivot2 = pivot1.reset_index()
mdf = melt(pivot2, id_vars=['Date'], value_vars=['딸기', '민트초코', '바닐라', '초콜릿'],
           var_name='Flavor', value_name='Quantity')

mdf

,Date,Flavor,Quantity
0,2023-07-01,딸기,18.000
1,2023-07-02,딸기,NaN
2,2023-07-03,딸기,25.000
3,2023-07-04,딸기,22.000
4,2023-07-01,민트초코,NaN
5,2023-07-02,민트초코,15.000
6,2023-07-03,민트초코,NaN
7,2023-07-04,민트초코,17.000
8,2023-07-01,바닐라,25.000
9,2023-07-02,바닐라,NaN


### 3. 게임 토너먼트 최고 득점자를 찾아라!
**game_scores 데이터 셋은 최근 열린 게임 토너먼트의 경기 결과 데이터이다.**   
- 데이터는 선수 이름(player)과 각 스테이지(stage1, stage2, stage3) 점수로 구성되어 있습니다.
- 일부 선수는 특정 스테이지에 참여하지 않아 점수가 비어있을 수 있습니다.
- 모든 스테이지에 참여한 선수들 중에서, 평균 점수가 가장 높은 "최고의 선수"를 찾아내고, 그 선수의 평균 점수를 계산하세요.   

**요구사항**
1. 각 스테이지 점수가 열로 구성된 데이터를, 선수와 스테이지 레벨에 따른 점수를 나타내는 형태로 재구조화하세요.
2. 선수별 평균 점수를 계산하세요.

In [22]:
# 1. 라이브러리 참조
from jussam import load_data
from pandas import DataFrame, pivot_table, melt

# 2. 데이터 불러오기
scores = load_data('game_scores')

# 3. 결측치 확인
na = scores.isna()
na.sum()

📚 최근 열린 게임 토너먼트 경기 결과 데이터 (인덱스와 메타데이터 없음)


player    0
stage1    0
stage2    1
stage3    1
dtype: int64

In [23]:
# 4. 데이터 정제 (모든 스테이지에 참여한 선수 데이터)
scores1 = scores.dropna().set_index('player')
scores1

,stage1,stage2,stage3
player,,,
Alice,85,92.000,88.000
Charlie,90,88.000,91.000
Eva,95,98.000,99.000


In [24]:
# 5. wide -> long
st1 = scores1.stack()
score_df = DataFrame(st1)

score_df

0
player               
Alice   stage1 85.000
        stage2 92.000
        stage3 88.000
Charlie stage1 90.000
        stage2 88.000
        stage3 91.000
Eva     stage1 95.000
        stage2 98.000
        stage3 99.000

In [25]:
avg_score = score_df.groupby('player').mean()
avg_score.sort_values(0, ascending=False)

,0
player,
Eva,97.333
Charlie,89.667
Alice,88.333
